# Fact-QA corpus-reliance dose-response — Colab runner

The **second, simulator-free** validation of the necessity instrument. Teach a set of
**fictional facts** into the weights, then show corpus-reliance (necessity = answer-accuracy
with the fact − without it) falls monotonically as the model memorizes them. Because the facts
are invented, a competent model MUST read the corpus at the naive end (necessity ≈ 1) and needs
it not at all once taught (≈ 0) — a clean known-groups curve with **no simulator**, just an
answer checker. Mirrors `DOSE_RESPONSE.md`; see `src/engine/factqa/README.md`.

Runtime → Change runtime type → **GPU** first.

## 1 · Confirm the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime → Change runtime type → GPU'

## 2 · Config + clone the repo

In [ ]:
import os

REPO_URL = 'https://github.com/wrgr/socratic-scenarios.git'
BRANCH   = 'main'                         # everything is merged to main; main never gets reset

MODEL    = 'Qwen/Qwen2.5-3B-Instruct'   # A100: 7B is easy in bf16 — 'Qwen/Qwen2.5-7B-Instruct' for a stronger, second-size run.
DTYPE    = 'bfloat16'
ALPHAS   = '0,0.25,0.5,0.75,1.0'        # the LoRA-α knowledge gradient (0 = naive, 1 = taught)
EPOCHS   = '8'                          # memorizing 25 arbitrary fictional facts needs repetition
LR       = '1e-4'
BATCH    = '4'                          # ~100 teach examples → ~25 steps/epoch
SAVE_EVERY = '15'                       # ≈6 checkpoints across the run for the cross-check
LORA_R   = '16'                         # more capacity for arbitrary fact associations than the r=8 default
# Factual memories live in the MLP layers, not attention; PEFT's default targets Q/V only, which is
# why arbitrary-fact teaching was weak. Target all linear layers (an A100 affords it).
LORA_TARGETS = 'q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj'
RELIANCE_THR = '0.15'                   # 'reliant' cutoff on the 0-1 accuracy axis (not the barrier 50)
LOAD_4BIT = '0'                         # A100: full bf16, NO quantization (4-bit noise blunts fact memorization + muddies low-alpha). '1' only if OOM.

%cd /content
# Clone if absent, then hard-sync to the branch tip so a re-run never runs stale code.
![ -d socratic-scenarios ] || git clone --depth 1 --branch $BRANCH $REPO_URL
!cd socratic-scenarios && git fetch --depth 1 origin $BRANCH && git reset --hard FETCH_HEAD
REPO = '/content/socratic-scenarios'; ARM = REPO + '/experiments/unlearning'
%cd $ARM
print(f'config: model={MODEL} dtype={DTYPE} alphas={ALPHAS} load_4bit={LOAD_4BIT}')

## 3 · Install deps (Python + the Node scorer)

In [ ]:
!pip -q install 'transformers>=4.40' 'peft>=0.11' 'accelerate>=0.30' 'safetensors>=0.4' 'bitsandbytes>=0.43'
# Colab ships an old torchao that PEFT's LoRA dispatch rejects; we don't use it — remove it.
!pip -q uninstall -y torchao 2>/dev/null; echo removed-torchao-if-present
# The instrument is TypeScript (npx tsx); install its Node deps once.
!cd $REPO && npm install --no-audit --no-fund --loglevel=error
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4 · Build the teach set (from the KB — single source of truth)
The fictional facts, several phrasings each. The **KB is the one source of truth**: the same
`buildKB()` the instrument scores also emits the teach set, so we never teach something the
checker doesn't grade.

In [ ]:
# KB_TEACH_DUMP makes the Node runner write the teach set; path is REPO-relative (npm runs at REPO).
!cd $REPO && KB_TEACH_DUMP=experiments/unlearning/data/factqa_teach.jsonl npm run --silent factqa:leakage
!wc -l $ARM/data/factqa_teach.jsonl && head -2 $ARM/data/factqa_teach.jsonl

## 5 · Teach the fictional facts into the weights (SFT)
`--save_every` also snapshots checkpoints for the second (cross-check) gradient.

In [ ]:
import subprocess
cmd = ['python','unlearn.py','--method','sft','--model',MODEL,'--dtype',DTYPE,
       '--sft_file','data/factqa_teach.jsonl','--epochs',EPOCHS,'--lr',LR,
       '--batch_size',BATCH,'--lora_r',LORA_R,'--lora_targets',LORA_TARGETS,'--save_every',SAVE_EVERY,'--chat','--out','out/factqa_taught']
if LOAD_4BIT=='1': cmd += ['--load_4bit']
subprocess.run(cmd, cwd=ARM, check=True)

## 5b · Diagnostic — did teaching take? (closed-book recall, no corpus)
Independent of the instrument: ask the fully-taught model three fact questions with NO corpus.
If it recalls them, teaching worked and the dose-response should move. If it says "I don't know"
or is wrong, teaching didn't take — raise EPOCHS / LORA_R and re-teach.

In [ ]:
import json, subprocess, os
probe = [
  ('What is the primary mineral extracted at Kervan Station?', 'veltricite'),
  ('Who is the station commander of Sable Reach?', 'Petra Vhlka'),
  ('How is Thorne Array powered?', 'a helium siphon'),
]
pp = os.path.join(ARM,'results','_recall_probe.jsonl'); os.makedirs(os.path.dirname(pp), exist_ok=True)
open(pp,'w').write('\n'.join(json.dumps({'prompt': f'Answer the question. Give the shortest exact answer.\n\nQuestion: {q}\nAnswer:'}) for q,_ in probe)+'\n')
op = os.path.join(ARM,'results','_recall_out.jsonl')
cmd = ['python','score_offline.py','--model',MODEL,'--dtype',DTYPE,'--adapter','out/factqa_taught','--alpha','1.0','--prompts',pp,'--out',op,'--max_new','24']
if LOAD_4BIT=='1': cmd += ['--load_4bit']
subprocess.run(cmd, cwd=ARM, check=True)
for (q,gold), line in zip(probe, open(op)):
    ans = json.loads(line)['completion']
    print(f'Q: {q}\n  taught model: {ans!r}\n  expected: {gold}\n')

## 6 · Sweep LoRA-α → the corpus-reliance dose-response
One taught adapter, evaluated at each α, scored by the **fact-QA** instrument (`--runner
factqa:leakage`). **Predicted: necessity falls monotonically** as α rises (the model has the
facts in-weights and needs the corpus less). Unlike the hazard, single-fact granularity makes
the aggregate curve smooth without any difficulty ladder. If it does *not* fall, the instrument
isn't measuring what we claim — a real result, reported, not hidden.

In [ ]:
import subprocess, os
cmd = ['python','dose_response.py','--model',MODEL,'--dtype',DTYPE,
       '--runner','factqa:leakage','--metric','regret','--reliance-threshold',RELIANCE_THR,
       '--adapter','out/factqa_taught','--alphas',ALPHAS,'--probes','all',
       '--out','results/dose_factqa_alpha']
if LOAD_4BIT=='1': cmd += ['--load_4bit']
subprocess.run(cmd, cwd=ARM, check=True)   # prints the ASCII curve
print('\n----- results/dose_factqa_alpha.csv -----')
print(open(os.path.join(ARM,'results/dose_factqa_alpha.csv')).read())

## 7 · (Optional) checkpoint gradient — cross-check the α curve
Training checkpoints (early = fact-naive, late = fact-knowing) are the more defensible graded
axis. If this curve agrees with the α curve, the monotonicity isn't a gradient-method artifact.

In [ ]:
import glob, os, subprocess
cks = sorted(glob.glob(os.path.join(ARM,'out/factqa_taught/ckpt-*')), key=lambda p:int(p.split('-')[-1]))
print('checkpoints:', [os.path.basename(c) for c in cks])
if len(cks) >= 2:
    cmd = ['python','dose_response.py','--model',MODEL,'--dtype',DTYPE,
           '--runner','factqa:leakage','--metric','regret','--reliance-threshold',RELIANCE_THR,
           '--checkpoints',','.join(cks),'--probes','all','--out','results/dose_factqa_ckpt']
    if LOAD_4BIT=='1': cmd += ['--load_4bit']
    subprocess.run(cmd, cwd=ARM, check=True)
    print('\n----- results/dose_factqa_ckpt.csv -----')
    print(open(os.path.join(ARM,'results/dose_factqa_ckpt.csv')).read())
else:
    print('too few checkpoints — lower SAVE_EVERY or raise EPOCHS/BATCH and re-teach')

## Done
Send back `results/dose_factqa_alpha.csv` (and the checkpoint CSV if you ran it). A monotone
fall in necessity is the second-domain, simulator-free known-groups validation.

Tie-in to the sufficiency claim: the two ends of this curve are the two sufficiency verdicts —
the naive model reads **CONTRIBUTING** (the corpus is necessary), the fully-taught model reads
**FALSE SUFFICIENCY** (it now answers from weights, so the corpus is redundant). The dose-response
is a controlled walk between them.